# MaterialMind-ECE — Phase 3: Feature Engineering & Selection
### Unit 2: Data & Preprocessing | Project 7: Electronic Material Clustering

This notebook implements **Phase 3: Feature Engineering and Feature Selection** on the 1,056 benchmark inorganic electronic and dielectric materials.

#### Core Objectives:
1. **Construct and evaluate scientifically grounded features**:
   - `ionic_polarization_fraction` = $\text{poly\_ionic} / \text{poly\_total}$
   - `refractive_index_squared` = $n^2$
2. **Check distributions, ranges, and invalid values** for all engineered candidates.
3. **Investigate multicollinearity and linear dependencies**:
   - The exact physical identity: $\text{poly\_total} = \text{poly\_electronic} + \text{poly\_ionic}$
   - Optical redundancy between $n$, $n^2$, and $\text{poly\_electronic}$
4. **Compute Variance Inflation Factors (VIF)** to validate non-redundancy.
5. **Finalize the optimal 6-feature clustering representation**.
6. **Fit and export `models/final_scaler.joblib`** and `data/processed/features_engineered.csv`.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
%matplotlib inline
print("Libraries loaded successfully.")

## 1. Load Cleaned Dataset
We load the staged benchmark dataset from Phase 2 (`data/processed/materials_cleaned.csv`).

In [2]:
data_path = '../data/processed/materials_cleaned.csv'
if not os.path.exists(data_path):
    data_path = 'data/processed/materials_cleaned.csv'

df = pd.read_csv(data_path)
print(f"Loaded materials: {df.shape[0]} rows x {df.shape[1]} columns")
df[['material_id', 'formula', 'band_gap', 'poly_total', 'poly_electronic', 'poly_ionic', 'n', 'density', 'volume']].head(5)

## 2. Feature Construction & Validation
### Feature 1: `ionic_polarization_fraction` ($f_{\text{ionic}}$)
$$f_{\text{ionic}} = \frac{\text{poly\_ionic}}{\text{poly\_total}} = \frac{\varepsilon_{\text{ionic}}}{\varepsilon_r} = 1 - \frac{\varepsilon_\infty}{\varepsilon_r}$$

> **Physical Meaning:** Quantifies the proportion of the static dielectric response that arises from **lattice phonon vibrations** (relative displacement of positive and negative sublattices) versus electronic electron-cloud distortion. It is **not** "dielectric loss"; it is the static polarization partition.

### Feature 2: `refractive_index_squared` ($n^2$)
$$\text{refractive\_index\_squared} = n^2$$
> **Physical Meaning:** Under Maxwell's electromagnetic relations for non-magnetic media ($\mu_r \approx 1$), optical permittivity is $\varepsilon_\infty = n^2$.

In [3]:
# 1. Construct ionic_polarization_fraction
df['ionic_polarization_fraction'] = df['poly_ionic'] / df['poly_total']

# Check invalid values
inv_ionic = df['ionic_polarization_fraction'].isnull().sum() + np.isinf(df['ionic_polarization_fraction']).sum()
print(f"ionic_polarization_fraction: Invalid values = {inv_ionic}")
print(f"Range: [{df['ionic_polarization_fraction'].min():.4f}, {df['ionic_polarization_fraction'].max():.4f}]")
print(f"Mean: {df['ionic_polarization_fraction'].mean():.4f}, Median: {df['ionic_polarization_fraction'].median():.4f}")

# 2. Construct refractive_index_squared
df['refractive_index_squared'] = df['n'] ** 2
inv_n2 = df['refractive_index_squared'].isnull().sum() + np.isinf(df['refractive_index_squared']).sum()
print(f"\nrefractive_index_squared: Invalid values = {inv_n2}")
corr_n2_elec = df['refractive_index_squared'].corr(df['poly_electronic'])
print(f"Correlation between n^2 and poly_electronic: {corr_n2_elec:.6f}")

## 3. Multicollinearity Investigation: $\text{poly\_total} = \text{poly\_electronic} + \text{poly\_ionic}$
We test the exact mathematical identity between total, electronic, and ionic dielectric constants.

In [4]:
residual = df['poly_total'] - (df['poly_electronic'] + df['poly_ionic'])
print("Difference between poly_total and (poly_electronic + poly_ionic):")
print(f"Mean difference: {residual.mean():.4e}")
print(f"Max absolute error: {np.abs(residual).max():.4e}")
print("Conclusion: poly_total is an EXACT linear sum of poly_electronic and poly_ionic!")

### Variance Inflation Factor (VIF) Analysis
We compute VIF across the initial 7 features to measure multicollinearity severity:
$$\text{VIF}_i = \frac{1}{1 - R_i^2}$$

In [5]:
def compute_vif(data, cols):
    X = StandardScaler().fit_transform(data[cols])
    vifs = {}
    for i, col in enumerate(cols):
        y_col = X[:, i]
        x_other = np.delete(X, i, axis=1)
        r2 = np.linalg.lstsq(x_other, y_col, rcond=None)[0]
        y_pred = x_other @ r2
        r2_val = 1.0 - np.sum((y_col - y_pred)**2) / np.sum(y_col**2)
        vif = 1.0 / (1.0 - r2_val) if (1.0 - r2_val) > 1e-10 else np.inf
        vifs[col] = vif
    return vifs

init_features = ['band_gap', 'poly_total', 'poly_electronic', 'poly_ionic', 'n', 'density', 'volume']
vif_init = compute_vif(df, init_features)
print("VIF for Initial 7 Features:")
for k, v in vif_init.items():
    print(f"  {k:20s}: {v:8.2f}")

## 4. Final Feature Selection & VIF Validation
We select 6 non-redundant, full-rank features:
1. `band_gap`: Primary electronic conductivity regime descriptor.
2. `poly_total`: Primary static permittivity figure of merit for ECE capacitors and gate dielectrics.
3. `poly_electronic`: Optical high-frequency response (replaces redundant $n$ and $n^2$).
4. `ionic_polarization_fraction`: Intrinsic scale-invariant mechanism ratio (replaces collinear `poly_ionic`).
5. `density`: Mass density / packing descriptor.
6. `volume`: Unit cell structural size.

In [6]:
final_features = [
    'band_gap',
    'poly_total',
    'poly_electronic',
    'ionic_polarization_fraction',
    'density',
    'volume'
]

vif_final = compute_vif(df, final_features)
print("VIF for Final Selected 6 Features:")
for k, v in vif_final.items():
    print(f"  {k:28s}: {v:8.2f}")

print("\nAll VIFs are strictly < 3.3. Infinite multicollinearity completely eliminated!")

## 5. Visualizations
We inspect the final feature correlation matrix and empirical feature distributions.

In [7]:
# Final Correlation Heatmap
plt.figure(figsize=(8, 6.5))
sns.heatmap(df[final_features].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('MaterialMind-ECE: Final Selected Features Correlation Heatmap', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

In [8]:
# Distributions of Final Features
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, feat in enumerate(final_features):
    sns.histplot(df[feat], kde=True, ax=axes[i], color='#2166ac' if feat != 'ionic_polarization_fraction' else '#1b7837', bins=30)
    axes[i].set_title(feat, fontweight='bold')
plt.suptitle('MaterialMind-ECE: Final Selected Clustering Feature Distributions', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Export Final Scaler & Processed Dataset
We fit a new `StandardScaler` on the 6 final clustering features and export artifacts.

In [9]:
# Fit final StandardScaler
scaler_final = StandardScaler()
X_scaled = scaler_final.fit_transform(df[final_features])
df_scaled = pd.DataFrame(X_scaled, columns=final_features, index=df.index)

# Export models/final_scaler.joblib
models_dir = '../models' if os.path.exists('../models') else 'models'
os.makedirs(models_dir, exist_ok=True)
final_scaler_file = os.path.join(models_dir, 'final_scaler.joblib')
joblib.dump(scaler_final, final_scaler_file)
print(f"Exported final scaler to: {final_scaler_file}")

# Export data/processed/features_engineered.csv
processed_dir = '../data/processed' if os.path.exists('../data/processed') else 'data/processed'
export_cols = ['material_id', 'formula'] + final_features
eng_csv_file = os.path.join(processed_dir, 'features_engineered.csv')
df[export_cols].to_csv(eng_csv_file, index=False)
print(f"Exported engineered features to: {eng_csv_file}")
print(f"Final feature matrix shape: {X_scaled.shape}")